# Image Enhancement
__Quantitative Big Imaging__ ETHZ: 227-0966-00L


<div class="rows">
    <div class="column23">
    
<p style="font-size:1em;">March 5, 2026</p>
<br /><br />
<p style="font-size:1.5em;padding-bottom: 0.25em;">Anders Kaestner</p>  
<p style="font-size:1em;">Laboratory for Neutron Scattering and Imaging<br />Paul Scherrer Institut</p>
</div>
        <div class="column13">
       <img src="../../docs/figures/np_photo-filters_2344219_000000.svg" style="height:200px"/>     
</div>
</dim>

## Todays lecture

- Imperfect images and noise
- Filters
- The Fourier transform
- Advanced filters
- Structure tensor
- Evaluation workflows

### Literature
#### Basic filters
- B. Jähne,[Digital Image Processing](https://doi.org/10.1007/3-540-27563-0), Springer Verlag, 2005.
- J.C. Russ,[The Image Processing Handbook](http://dx.doi.org/10.1201/9780203881095), Wiley, 2006.

#### Orientations
- Z. Püspöki et al., [Transforms and Operators for Directional Bioimage Analysis: A Survey](https://doi.org/10.1007/978-3-319-28549-8_3) in Focus on Bio-Image Informatics, Springer Verlag, 2016.
- J. Bigün [Vision with Direction](https://doi.org/10.1007/b138918), Springer Verlag, 2006.

[Orientation demonstration](http://bigwww.epfl.ch/demo/ip/demos/orientation/)

#### Metrics
Wang and Bovik, [Mean squared error: Love it or leave it? A new look at Signal Fidelity Measures](https://doi.org/10.1109/MSP.2008.930649), IEEE Signal Processing Magazine, 2009

... and some more detailed publications during the lecture.

### We need some modules

These modules are needed to run the python cells in this lecture.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib        as mpl
from matplotlib.colors import LogNorm
import skimage as ski
import skimage.filters as flt
import skimage.io as io
from skimage.morphology import disk
import scipy.ndimage as ndimage
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter
from matplotlib.patches import ConnectionPatch
from scipy.signal import medfilt

mpl.rcParams['figure.dpi'] = 100

# Motivation - Measurements are rarely perfect
<figure><img src="figures/imperfect_imaging_system.svg" style="height:500px" align="middle"></figure>

There is no perfect measurement. This is also true for imaging. The ideal image of the sample is distorted for many reasons. The figure below shows how an image of a sample can look after passing though the acquisition system. These quailty degradations will have an impact on the analysis of the image data. In some cases, it is possible to correct for some of these artifacts using traditional image processing techniques. There are however also cases that require extra effort, including implementing new algorithms, to correct the artifacts.  

```{figure} figures/imperfect_imaging_system.png
:width: 14cm

Schematic showing the error contributions in an imperfect imaging system.
```

## Factors affecting the image quality

No measurement system is perfect and therefore, you will also not get perfect images of the sample you measured. In the figure you can see the slice as it would be in a perfect world to the left and what you actually would measure with an imaging system. Some are of higher quality than others but there are still a collection of factors that have an impact on the image quality.

The list below provides some factors that affect the quality of the acquired images. Most of them can be handled by changing the imaging configuration in some sense. It can however be that the sample or process observed put limitiations on how much the acquisition can be tuned to obtain the perfect image.

* Resolution (Imaging system transfer functions)
* Noise
* Contrast
* Inhomogeneous contrast
* Artifacts

### Resolution
The resolution is primarily determined optical transfer function of the imaging system. The actual resolution is given by the extents of the sample and how much the detector needs to capture in one image. This gives the field of view and given the number pixels in the used detector it is possible to calculate the pixel size. The pixel size limits the size of the smallest feature in the image that can be detected. The scintillator, which is used to convert neutrons into visible light, is chosen to 
1. match the sampling rate given by the pixel size.
2. provide sufficient neutron capture to obtain sufficient light output for a given exposure time.

### Noise
An imaging system has many noise sources, each with its own distribution e.g.
1. Neutron statistics - how many neutrons are collected in a pixel. This noise is Poisson distributed. 
2. Photon statistics - how many photons are produced by each neutron. This noise is also Poisson distributed.
3. Thermal noise from the electronics which has a Gaussian distribution.
4. Digitation noise from converting the charges collected for the photons into digital numbers that can be transfered and stored by a computer, this noise has a binominal distribution.

The neutron statistics are mostly dominant in neutron imaging but in some cases it could also be that the photon statistics play a role. 

### Contrast
The contrast in the sample is a consequence of 
1. how well the sample transmits the chosen radiation type. For neutrons you obtain good contrast from materials containing hydrogen or lithium while many metals are more transparent.
2. the amount of a specific element or material represented in a unit cell, e.g. a pixel (radiograph) or a voxel (tomography). 

The objective of many experiments is to quantify the amount of a specific material. This could for example be the amount of water in a porous medium.

Good contrast between different image features is important if you want to segment them to make conclusions about the image content. Therefore, the radiation type should be chosen to provide the best contrast between the features.

### Inhomogeneous contrast
The contrast in the raw radiograph depends much on the beam profile. These variations are easily corrected by normalizing the images by an open beam or flat field image. 

- __Biases introduced by scattering__ Scattering is the dominant interaction for many materials use in neutron imaging. This means that neutrons that are not passing straight though the sample are scattered and contribute to a background cloud of neutrons that build up a bias of neutron that are also detected and contribute to the 

- __Biases from beam hardening__ is a problem that is more present in x-ray imaging and is caused by that fact that the attenuation coefficient depends on the energy of the radiation. Higher energies have lower attenuation coefficient, thus will high energies penetrate the thicker samples than lower energies. This can be seen when a polychromatic beam is used. 

#### Artifacts
Many images suffer from outliers caused by stray rays hitting the detector. Typical artefacts in tomography data are
- Lines, which are caused by outlier spots that only appear in single projections. These spot appear as lines in the reconstructed images.
- Rings are caused by stuck pixels which have the same value in a projections.

## A typical processing chain

Traditionally, the processing of image data can be divided into a series of subtasks that provide the final result.

```{figure} figures/image_proc_chain.png
:width: 12cm

Typical steps of an image processing work flow.
```

* __Acquisition__ The data must obviously be acquired and stored. There are cases when simulated data is used. Then, the acquisition is replaced by the process to simulate the data.
* __Enhancement__ The raw data is usually not ready to be processed in the form is comes from the acquisition. It usually has noise and artifacts as we saw on the previous slide. The enhancement step suppresses unwanted information in the data.
* __Segmentation__ The segmenation identifies different regions based on different features such as intensity distribution and shape.
* __Post processing__ After segmentation, there may be falsely identified regions. These are removed in a post processing step. 
* __Evaluation__ The last step of the process is to make conclusions based on the image data. It could be modelling material distrbutions, measuring shapes etc.

<figure><img src="figures/image_proc_chain.svg" style="height:300px" align="middle"></figure>

Today's lecture will focus on __enhancement__

# Noise and artifacts

Noise is in very general terms for the unwanted information in a signal.

More specifically;  

We are talking about 
- __random contributions__ that
- __obscure the image information__ we are interested in.

## Noise types

Noise can have many different characteristics. In general, it is driven by a random distribution.

* Spatially uncorrelated noise

With spatially uncorrelated noise each pixel has a random value which is independend of the pixel neighborhood. This is also the easiest noise type to simulate.

* Event noise

The even noise has a random activation function that triggers the event of each pixel with some probabilty. This noise type produces spots randomly distributed over the image. The spots may also have a randomw intensity.

* Stuctured noise

The structured noise depends on the values of the pixel neighborhood and is thus spatially correlated. It is mostly driven by an uncorrelated noise source which is blurred by a weighted combination of the neighborhood. 

An example of structured noise is when photons arrive at a scintillator and they produce a glowing spot which has greater spatial extent than a pixel.

The figure below shows examples of the three noise types.

In [ ]:
plt.figure(figsize=[12,4]);
plt.suptitle('Noise examples')
plt.subplot(1,3,1);plt.imshow(np.random.normal(0,1,[100,100])); plt.title('Gaussian');plt.axis('off');
plt.subplot(1,3,2);plt.imshow(0.90<np.random.uniform(0,1,size=[100,100]),cmap='gray'); plt.title("Salt and pepper"),plt.axis('off');
plt.subplot(1,3,3);plt.imshow(ski.filters.gaussian(np.random.normal(0,1,size=[100,100]),sigma=1),cmap='gray'); plt.title("Structured"),plt.axis('off');
plt.tight_layout()

### Noise models - Gaussian noise

Gaussian noise is the most common random distribution used. All other distributions asymptotically converges towards the Gaussian distribution thanks to the [central limit theorem](https://en.wikipedia.org/wiki/Central_limit_theorem). The Gaussian noise is an easy distribution to work with when you derive signal processing models. This is also the reason why it is so popular to use this model also for non-Gaussian noise.

* Additive
* Easy to model 
* Law of large numbers

__Distribution function__

$$n(x)=\frac{1}{\sqrt{2\pi\sigma}}\exp{-\left(\frac{x-\mu}{2\sigma}\right)^2}$$

Below you see plots of the Gaussian distribution with different parameters.

In [ ]:
from scipy.stats import norm
rv = norm(loc = -1., scale = 1.0);rv1 = norm(loc = 0., scale = 2.0); rv2 = norm(loc = 2., scale = 3.0)
x = np.arange(-10, 10, .1)
plt.figure(figsize=(5,3))
#plot the pdfs of these normal distributions 
plt.plot(x, rv.pdf(x),label='$\mu$=-1, $\sigma$=1')
plt.plot(x, rv1.pdf(x),label='$\mu$=0, $\sigma$=2') 
plt.plot(x, rv2.pdf(x),label='$\mu$=2, $\sigma$=3')
plt.legend();

### Noise models - Poisson noise

The Poisson noise is the central noise model for event counting processes. It is thus the type of noise you see in imaging as the detectors in some sense is counting the number of particles arriving at the detector, e.g. photons or neutrons. This noise distribution changes shape with increasing number of particles; the distribution is clearly asymmetric for few particles while it takes a Gaussian shape when many particles are counted. It is also multiplicative in contrast to the Gaussian noise. This is in other words the noise distribution you need to use if you want to model image noise correctly.

* Intensity dependent 
* Physically correct for event counting

__Distribition function__ 

$$p(x)=\frac{\lambda^{k}}{k!} e^{-\lambda\,x}$$

The plot below show a poisson distributions for $\lambda$=3 and $\lambda$=10000.

In [ ]:
from scipy.stats import poisson
mu=3
fig, ax = plt.subplots(1, 2, figsize=[10,4])
x = np.arange(poisson.ppf(0.01, mu),              poisson.ppf(0.999, mu))
ax[0].plot(x, poisson.pmf(x, mu), 'bo', ms=8, label='poisson pmf')
ax[0].vlines(x, 0, poisson.pmf(x, mu), colors='b', lw=5, alpha=0.5);
ax[0].set_title(r"$\lambda$={0:0.1f}".format(mu));

mu=10000
x = np.arange(poisson.ppf(0.005, mu),              poisson.ppf(0.999, mu),20)
ax[1].plot(x, poisson.pmf(x, mu), 'bo', ms=8, label='poisson pmf')
ax[1].vlines(x, 0, poisson.pmf(x, mu), colors='b', lw=5, alpha=0.5);
ax[1].set_title(r"$\lambda$={0:0.1f}".format(mu));

### Compare Gaussian and Possion noise

Now, let's compare relaizations of Gaussian and Poisson noise overlaid on a sine curve. The important thing to observe is that the noise amplitude is independent of the signal value and constant for the Gaussian noise. For Poisson noise it is very different. The noise amplitude changes with the signal values. Higher signal strength also produces greater noise amplitudes.

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(12,7))
ax=ax.ravel()

x=np.linspace(0,2*np.pi,2000)
y=1*np.sin(x)+2

idxmax = np.argmax(y)
idxmin = np.argmin(y)


np.random.seed(42)
ng=np.random.normal(0,0.5,size=len(x))
gn=y+ng
ax[0].plot(x,gn,'.',label='Gaussian noise',alpha=0.5)
ax[0].plot(x,y,label='Original',lw=2)
ax[0].set_ylim([-1,5])
ax[0].set_title('Gaussian noise')
stdmin = gn[(idxmin-100):(idxmin+100)].std()
stdmax = gn[(idxmax-100):(idxmax+100)].std()
ax[0].plot([x[idxmin],x[idxmin]],[y[idxmin]+stdmin,y[idxmin]-stdmin],lw=5,c='red',label=r"$\mu \pm \sigma$")
ax[0].plot([x[idxmax],x[idxmax]],[y[idxmax]+stdmax,y[idxmax]-stdmax],lw=5,c='red')
ax[0].legend()
ax[0].axis('off')

pn=np.random.poisson(y*20)/20

ax[1].plot(x,pn,'.',label='Poisson noise',alpha=0.5)
ax[1].plot(x,y,label='Original',lw=2)
ax[1].set_ylim([-1,5])
ax[1].set_title('Poisson noise');
ax[1].axis('off')

stdmin = pn[(idxmin-100):(idxmin+100)].std()
stdmax = pn[(idxmax-100):(idxmax+100)].std()
ax[1].plot([x[idxmin],x[idxmin]],[y[idxmin]+stdmin,y[idxmin]-stdmin],lw=5,c='red',label=r"$\mu \pm \sigma$")
ax[1].plot([x[idxmax],x[idxmax]],[y[idxmax]+stdmax,y[idxmax]-stdmax],lw=5,c='red')
ax[1].legend();

ax[2].plot(x,ng);ax[2].axis('off');
ax[3].plot(x,pn-y);ax[3].axis('off');
plt.suptitle('Samples of two distributions');

### Noise models - Salt'n'pepper noise
* A type of outlier noise
* Noise frequency described as probability of outlier
* Can be additive, multiplicative, and independent replacement

__Example model__
$$sp(x)=\left\{\begin{array}{ll}
-1 & x\leq\lambda_1\\ 
0 & \lambda_1< x \leq \lambda_2\\
1 & \lambda_2<x
\end{array}\right.\qquad \begin{array}{l}x\in\mathcal{U}(0,1)\\\lambda_1<\lambda_2\\
\lambda_1+\lambda_2 = \mbox{noise fraction}
\end{array}$$

### Salt'n'pepper examples

In [ ]:
def snp(dims,Pblack,Pwhite) : # Noise model function
    uni=np.random.uniform(0,1,dims)
    img=(Pwhite<uni).astype(float)-(uni<Pblack).astype(float)
    return img

In [ ]:
img10_90=snp([100,100],0.1,0.9); img5_95=snp([100,100],0.05,0.95);img1_99=snp([100,100],0.01,0.99)
plt.figure(figsize=[15,5])
plt.subplot(1,3,1); plt.imshow(img1_99,cmap='gray'); plt.title('$\lambda_1$=1% and $\lambda_2$=99%',fontsize=16); plt.axis('off');
plt.subplot(1,3,2); plt.imshow(img5_95,cmap='gray'); plt.title('$\lambda_1$=5% and $\lambda_2$=95%',fontsize=16); plt.axis('off');
plt.subplot(1,3,3); plt.imshow(img10_90,cmap='gray'); plt.title('$\lambda_1$=10% and $\lambda_2$=90%',fontsize=16); plt.axis('off');

## Signal to noise ratio

It is important to know how strong the noise is compared to the signal in order to decide how to proceed with the analysis. Therefore, we need a metric to quantify the noise. 

The Signal to noise ratio measures the noise strengh in a signal

__Definition__

$$SNR=\frac{mean(f)}{stddev(f)}$$

Sometimes the term contrast to noise ratio is also used. This means that you measure the intensity difference in between two relevant features and divide this by the noise.

### Signal to noise ratio for Poisson noise

The SNR of poisson noise is particularly easy to compute because $E[x]=v[x]$. This means that the SNR is proportional to the square root of the number of particles. 

- For a Poisson distribution the SNR is :

$$SNR=\frac{E[x]}{s[x]}\sim\frac{N}{\sqrt{N}}=\sqrt{N}$$

- $N$ is the number of particles $\sim$ exposure time

where _N_ is the number of captured particles. The figure below shows two neutron images acquired at 0.1s and 10s respectively. The plot shows the signal to noise ratio obtained for different exposure times.

The signal to noise ratio can be improved by increasing the number of neutrons per pixel. This can be achived through increasing
- Flux - this is usually relatively hard alter as the radiation sources operate with the parameters it is designed for. There is a posibilty by changing the aperture, but this has an impact of the beam quality. 
- Exposure time - the exposure time can be increased but in the end there is a limitation on how much this can be used. Beam time is limited which means the experiment must be finished in a given time. There is also an upper limit on the exposure time defined by the observed sample or process when it changes over time. Too long exposure times will result in motion artefacts.
- Pixel size - increasing the pixel size means that photons or neutrons are collected over a greater area and thus more photons or neutrons are captured during the exposure. The limit on how much you can increase the pixel size is defined by the smallest features you want to detect.
- Detector material and thickness - the number of captured neutrons depends on the scintillator material and how thick it is. The thickness does however have an impact on the resolution. Therefore scintillator thickness and pixel size often increase in parallel as there is no point in oversampling a smooth signal to much.

In the end, there are many parameters that combined results in the SNR you obtain. These parameters are tuned to match the experiment conditions. The filtering techniques presented in this lecture can help to increase the SNR and hopefuly make the way for a quantitaive analysis.

In [ ]:
exptime=np.array([50,100,200,500,1000,2000,5000,10000])
snr = np.array([ 8.45949767, 11.40011621, 16.38118766, 21.12056507, 31.09116641,40.65323123, 55.60833117, 68.21108979]);
marker_style = dict(color='cornflowerblue', linestyle='-', marker='o',markersize=10, markerfacecoloralt='gray');
fig,ax = plt.subplots(1,3,figsize=(15,4)) 

ax[1].plot(exptime/1000,snr, **marker_style);
ax[1].set_xlabel('Exposure time [s]');
ax[1].set_ylabel('SNR [1]')
img50ms    = plt.imread('figures//tower_50ms.png'); 
img10000ms = plt.imread('figures/tower_10000ms.png');

ax[0].imshow(img50ms); 
ax[0].set_title('50ms'); 

ax[2].imshow(img10000ms)
ax[2].set_title('10s');

con0 = ConnectionPatch(xyA=(450,700), xyB=(exptime[0]/1000,snr[0]), 
                       coordsA="data", coordsB="data", 
                       axesA=ax[0], axesB=ax[1], 
                       color="crimson", lw=3)
ax[1].add_artist(con0)

con2 = ConnectionPatch(xyA=(0,200), xyB=(exptime[-1]/1000,snr[-1]), 
                       coordsA="data", coordsB="data", 
                       axesA=ax[2], axesB=ax[1], 
                       color="crimson", lw=3)
ax[1].add_artist(con2);

## Some useful python functions

### Random number generators [numpy.random]
Generate an $m \times n$ random fields with different distributions:
* __Gauss__ ```np.random.normal(mu,sigma, size=[rows,cols])```
* __Uniform__ ```np.random.uniform(low,high,size=[rows,cols])```
* __Poisson__ ```np.random.poisson(lambda, size=[rows,cols])```
	
### Statistics

* ```np.mean(f)```, ```np.var(f)```, ```np.std(f)``` Computes the mean, variance, and standard deviation of an image $f$.
* ```np.min(f)```,```np.max(f)``` Finds minimum and maximum values in $f$.
* ```np.median(f)```, ```np.rank()``` Selects different values from the sorted data.


# Basic filtering

## What is a filter?

In general terms a filter is a device that separates mixed components form each other. In chemistry you use a filter to separate solid patricles from liquid. In signal processing the filter is used to separate frequencies in signals. In image processing we are not only talking about frequencies but also structures. This is something we will look into in a few weeks when we talk about morphological image processing.

### General definition
A filter is a processing unit that
* Enhances the wanted information 
* Suppresses the unwanted information


> Ideally without altering relevant features beyond recognition .


The filter should ideally perform the task it is meant to do, but at the same time maintain the information we want to keep. This is often a difficult problem, in particular with traditional filters. They apply the same characteristics to any pixel without concerning what this pixel actually represents. It only sees frequencies! As a consequence you may cancel all relevant information in the image in your mission to remove the noise.

## Filter characteristics

Filters are characterized by the type of information they suppress or amplify.

In signal and image processing filters are applied to modify the amplitudes of different frequencies in the signal. Slow variations have low frequencies while rapid variations have high frequencies.

### Low-pass filters

Low pass filters are designed to suppress frequencies in the upper part of the spectrum in order to better show slow changes in the images. 

The effect of applying a lowpass filter is that the images are blurred.

```{figure} figures/lp_principle.png
:width: 10cm

The principle of a low-pass filter.
```

* Slow changes are enhanced 
* Rapid changes are suppressed 
<figure><img src="figures/lp_principle.svg" style="height:300px" align="middle"></figure>

### High-pass filters

High pass filters are the opposite of the low pass filters as the name suggests. The suppress low frequency components of the spectrum leaving the rapid changes untouched. This is an important filter for edge detection as we will see later.

```{figure} figures/hp_principle.png
:width: 10cm

The principle of a high-pass filter.
```

* Rapid changes are enhanced 
* Slow changes are suppressed
<figure><img src="figures/hp_principle.svg" style="height:300px" align="middle"></figure>

# Basic filters

## Linear filters
Computed using the convolution operation

$$g(x)=h*f(x)=\int_{\Omega}f(x-\tau) h(\tau) d\tau$$
where

* __$f$__ is the image 
* __$h$__ is the convolution kernel of the filter

<figure><img src="figures/filter_box.svg" style="height:150px" align="middle"></figure>

[Convolution on 1Brown3Blues](https://youtu.be/KuXjwB4LzSA?si=f2dTfiBHh0CLrt7l)

You can watch the movie produced by 1Brown3Blues if you want a visual demonstration of the convolution.

```{figure} figures/filter_box.png
:width: 6cm

Box plot of a the signal f convolved by the transfer function h.
```

The convolution above is defined for continuous signals. Images are however discrete signals as we saw in the first lecture. Thus we also have to sample the convolution kernel into a discrete filter window. The filter is typically a small grid that is sufficiently large to represent the convolution kernel.

## Low-pass filter kernels

The most common linear filters used in image processing are the box filter and the Gauss filter. The box filter has a kernel where all filter weights have the same strength. This filter essentially computes the local average of the neighborhood it covers. 
A 5x5 box filter looks like this

$$B=\frac{1}{25}\cdot\begin{array}{|c|c|c|c|c|}
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
\end{array}
$$ 

The scaling by the number of weights is sometimes omitted, but this would mean that the intensity of the resulting image is upscaled by this factor.

The Gauss filter kernel has its weights from the N-Dimensional Gauss function. Here, a 2D kernel:

$$G(x,y)=e^{-\frac{x^2+y^2}{2\sigma^2}}$$

The Gauss function is a continuous function that extends to infinity. Therefore, we have to define the size of the discrete kernel. A good choise is $N=2\cdot\lceil 2 \sigma \rceil+1$, multiple of $\sigma$ can also be set to 2.5 or even 3 but that is almost too much because the boundary weights are very small compared to the central value. 

```{figure} figures/gauss_bell.png
:width: 8cm

The shape of a Gaussian filter kernel.
```

<table>
<tr style="bgcolor:#FFFFFF"><td>Mean or Box filter</td><td>Gauss filter</td></tr>
    
<tr><td>
   All weights have the same value.
</td><td>
    
$$G(x,y)=e^{-\frac{x^2+y^2}{2\,\sigma^2}}$$

</td></tr>
    
<tr><td>
    
Example:
$$B=\frac{1}{25}\cdot\begin{array}{|c|c|c|c|c|}
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
1 & 1 & 1 & 1& 1\\
\hline
\end{array}
$$   
    
</td>
<td>

Example:
<figure><img src="figures/gauss_bell.svg" style="height:300px" align="middle"></figure>       
</td></tr>    
</table>


<div class="alert alert-block alert-success">
<center>Low-pass filters suppress noise</center>
</div>

It also important to note that the filter kernel should preferably have an odd number of elements. This is needed to avoid shifting the image by one or more pixels. The center pixel is obvious for odd number of kernel elements. The position is however not clear for an even number, it would at least be shifted by 1/2 pixel from the true center. 

## Different SNR using a Gauss filter

The main purpose of lowpass filters is to reduce the noise in the images. In the following example you can see images with different SNR and what happens when you apply Gauss filters with different $\sigma$.

In [ ]:
fig, ax = plt.subplots(3,4,figsize=(15,10)); ax=ax.ravel()
img   = plt.imread('figures/input_orig.png'); 
noise = np.random.normal(0,1,size=img.shape); 
SNRs   = [1000,10, 5, 2]
sigmas = [0.001,1,3]
for r,sigma in enumerate(sigmas) :
    for c,SNR in enumerate(SNRs) :
        ax[r*(len(SNRs))+c].imshow(ski.filters.gaussian(img+noise/SNR,sigma=sigma),cmap='gray');
    ax[r*(len(SNRs))].set_ylabel('$\sigma$={0}'.format(sigma))
for c,SNR in enumerate(SNRs) :
    ax[c].set_title('SNR={0}'.format(SNR))

What you see in the example is that you will need a wider filter kernel to reduce noise in images with low SNR. The cost of the SNR improvement is unfortunately that fine details in the image are also blurred by the operation. From this example we can conclude that linear filters can't be applied with out consequences for the image content. Therefore, we have to carefully select filter kernel balancing the improvement in SNR against loss of image features.

## Resolution loss with low pass filtering

We saw in lecture 1 that there are many factors contributing to the resolution. Filtering the image was not included in the image formation model.
```{figure} figures/traditional-image-flow.png
:width: 12cm

Illustrating a traditional image acquisition flow (lecture 1).
```

<figure><img src="figures/traditional-image-flow.png" style="height:200px" align="middle"></figure>

The resolution is characterized by the Modulation Transfer Function (MTF)

$$MTF_{total}=MTF_{system}\cdot{}MTF_{filter}$$

For a Gaussian filter:

$$MTF_{filter}(f)=e^{-2\pi^2\sigma^2 f^2}$$ 

i.e. the effective resolution worsens.

> Filtering acts like increasing the scintillator thickness.

### Small details - The effect of increasing the kernel size

We have already seen that changing the size of the filter kernel blurrs the image, but what does it mean when you want to detect tiny features in the image?

In the following example we have an image with disks with different radii and apply a Gaussian filter with increasing $\sigma$. The plot shows the intensity profile at the center of the disks.

In [ ]:
import numpy as np
from skimage.draw import disk
import skimage.filters as flt

# Disk radii (pixels)
radii = np.array([1, 2, 5,8,10,15 ,20], dtype=int)

# Layout parameters
gap = 20         # minimal empty pixels between neighboring disk boundaries
margin = 20      # border margin to image edge
bg = 0
fg = 255

# --- Compute spacing to guarantee no overlap ---
# Need: dx >= r_i + r_{i+1} + gap for all neighbors
dx = int(np.max(radii[:-1] + radii[1:] + gap))

# Image height: fit the largest disk plus margins
Rmax = int(radii.max())
height = int(2 * (Rmax + margin))

# Image width: place N disks with spacing dx, plus margins and disk extents
N = len(radii)
width = int(2 * (Rmax + margin) + (N - 1) * dx)

# Centers
y0 = height // 2
x0 = Rmax + margin
xs = x0 + np.arange(N) * dx

# --- Draw ---
img = np.full((height, width), bg, dtype=np.uint8)
for x, r in zip(xs, radii):
    rr, cc = disk((y0, x), r, shape=img.shape)
    img[rr, cc] = fg



sigmas = [0,1,2,5,10]

fig,axs=plt.subplots(len(sigmas),2,figsize=[14,7])

for ax,sigma in zip (axs,sigmas) :
    fimg = flt.gaussian(img,sigma)
    ax[0].imshow(fimg)
    ax[0].axhline(height//2,color='r',linewidth=0.5)
    ax[0].set(ylabel=r"$\sigma$={0}".format(sigma),xticks=[],yticks=[])
    ax[1].plot(fimg[height//2,:])
    ax[1].set(yticks=[])

The first row is the original image as reference. You can observe two things in this sequence
1. The amplitude decreases radiacally when the kernel size increases. The smallest two even vanish for large kernels.
2. The width of the items increase. In the extreme with large kernels this even has the effect that the objects merge. This is particularly visible for $\sigma$=10.

Both observations have an effect on our ability to perform a quantitative analysis.

## How is the convolution computed

Before, we saw that the convolution was comouted using an integral. This is however the definition for continuous variables. In image processing, we change the integral into a sum instead. The convolution is then a weighted sum of the neighborhood pixels. The example below shows how a pixel is updated using a box kernel with the size 3x3. This operation is repeated for all pixels or voxels in the image.

```{figure} figures/principle_mean_filter.png
:width: 12cm

Updating one pixel using a 3x3 box filter.
```

<figure><img src="figures/principle_mean_filter.svg" style="height:400px" align="middle"></figure>


> For a non-uniform kernel each term is weighted by its kernel weight.


## Euclidean separability

The convolution involves quite many additions and multiplications to perform. This can be a bottleneck in a complicated processing workflow. Fortunately, some tricks can be used to recude the number of operations needed for the convolution.

The asociative and commutative laws apply to convoution

$$(a * b)*c=a*(b*c) \quad \mbox{ and } \quad a * b = b * a $$

A convolution kernel is called _separable_ if it can be split in two or more parts.

This is the case for the two filter kernels we have seen until now:

### Examples of separable kernels
__Box__

$$\begin{array}{|c|c|c|}
\hline\cdot&\cdot&\cdot\\
\hline\cdot&\cdot&\cdot\\
\hline\cdot&\cdot&\cdot\\
\hline\end{array}=\begin{array}{|c|}\hline\cdot \\
\hline\cdot\\
\hline\cdot\\
\hline\end{array}*
\begin{array}{|c|c|c|}\hline\cdot&\cdot&\cdot\\\hline\end{array}$$
    
__Gauss__

$$\exp{\left(-\frac{x^2+y^2}{2\sigma^2}\right)}=\exp{\left(-\frac{x^2}{2\sigma^2}\right)}*\exp{\left(-\frac{y^2}{2\sigma^2}\right)}$$

### Gain of using separable kernels

Let's see what it brings to split the kernels in to the principal directions of the image. 

Separability reduces the number of computations $\rightarrow$ faster processing

#### Let's try with N=3
- 3$\times$3 $\rightarrow$ 9 mult and 8 add $\Leftrightarrow$ 6 mult and 4 add
- 3$\times$3$\times$3 $\rightarrow$ 27 mult and 26 add $\Leftrightarrow$ 9 mult and 6 add

A moderate gain.

#### Now about N=5?
- 5$\times$5 $\rightarrow$ 25 mult and 24 add $\Leftrightarrow$ 10 mult and 8 add
- 5$\times$5$\times$5 $\rightarrow$ 125 mult and 124 add $\Leftrightarrow$ 15 mult and 12 add

It starts to pay off!

This looks very promising and it would be great. There are however some things to consider:
- Overhead to call the filter function may consume some of the time gain from the separability.
- The result may deviate a little depending on the used numerical precision used. 

## The median filter

The median filter is a non-linear filter with low-pass characteristics. This filter computes the local median using the pixels in the neighborhood and uses this value in the filtered image. 

```{figure} figures/principle_median_filter.png
:width: 12cm

Updating a pixel from its neighborhood using a 3x3 median filter.
```

<figure><img src="figures/principle_median_filter.svg" style="height:400px" align="middle"></figure>

## Comparing filters for different noise types

Both linear low-pass filters and the median filter have SNR improving charactersitics. They do, however, differ in which noise types they are suited for. In the example below you see an image with add Gaussian noise and salt'n'pepper noise. 

In [ ]:
img = plt.imread('figures/grasshopper.png'); 
noise=img+np.random.normal(0,0.1,size=img.shape); 
spots=img+0.2*snp(img.shape,0,0.8); 
noise=(noise-noise.min())/(noise.max()-noise.min());
spots=(spots-spots.min())/(spots.max()-spots.min());

# Visualization
fig,ax=plt.subplots(2,3,figsize=[15,10]); 
ax=ax.ravel()
(vmin,vmax)=(0.0,0.9);
cmap='gray'
ax[0].imshow(spots,vmin=vmin,vmax=vmax,cmap=cmap,interpolation='none'); 
ax[0].set(title='Spots', xticks=[],yticks=[]); 

ax[1].imshow(ski.filters.gaussian(spots,sigma=1),vmin=vmin,vmax=vmax,cmap=cmap,interpolation='none'); 
ax[1].set(title='Gauss filter', xticks=[],yticks=[]); 

ax[2].imshow(ski.filters.median(spots,disk(3)),vmin=vmin,vmax=vmax,cmap=cmap); 
ax[2].set(title='Median', xticks=[],yticks=[]); 

ax[3].imshow(noise,vmin=vmin,vmax=vmax,cmap=cmap); plt.title('Gaussian noise'); 
ax[3].set(title='Gaussian noise', xticks=[],yticks=[]); 

ax[4].imshow(ski.filters.gaussian(noise,sigma=1),vmin=vmin,vmax=vmax,cmap=cmap); 
ax[4].set(title='Gauss filter', xticks=[],yticks=[]); 

ax[5].imshow(ski.filters.median(noise,disk(3)),vmin=vmin,vmax=vmax,cmap=cmap); 
ax[5].set(title='Median', xticks=[],yticks=[]); 

In this comparison, you can clearly see that the median filter is supperior when it comes to remove outliers in an image. In this case, the Gauss filter is only smearing out the outliers, but they still appear as noisy. 

In the case of Gaussian noise, it is harder to tell which filter to use. Many are using the median filter as their main goto choise to filter images. Because it is usually gentler to edges and is also good at removing spots. Still, you should be aware of the additional processing time and also that the statistical distribution of the data is not following the original model anymore. The distribution of the data may not be of any concern in many cases, but if you are trying to process the image further based on its distribution, this may result in less good results.

### Filter example: Spot cleaning

```{figure} figures/spotty_knot_closeup.png
:width: 8cm

A neutron image with outliers that we want to remove.
```

<div class="row">
    <div class="column23">

#### The problem
        

The interaction with neutrons is either scattering or absorption. A reaction of the absorption is that a gamma radiation is emitted. These gamma photons may hit the detector with the consequences that

        
- Many neutron images are corrupted by spots that confuse following processing steps. 
- The amount, size, and intensity varies with many factors.   

#### Our filter options
        

We will now compare some options to suppress or remove the gamma spot from the radiography images. We have already seen the first two options. The last option is often used for outlier removal

<div>
<div class="column23">
    
- Low pass filter
- Median filter
- Detect spots and replace by estimate
        
</div>
    <div class="column13">
         <figure><img src="figures/spotty_knot_closeup.png" style="height:400px" align="middle"></figure> 
    </div>
</div>    

### A spot cleaning algorithm

The spot cleaning algorithm we will use in this example involves three steps.
- Enhancing the spots - this is achieved with a high pass filter which is implemented as the difference between the original image and the median filtered image (a low pass filter). The effect is that we will keep the high frequency components of the image. 
- Detecting the spots - We detect the spots by checking if the pixels in the high pass filtered image exceed a threshold. This produces a detection mask. 
- Replacing spots with new information - In this algorithm we implement the replacement as the sum of the spot mask multiplied by the median filtered image and the complement of the mask multiplied by the original image.

```{figure} figures/spotclean_algorithm.png
:width: 14cm

Process workflow to selectively remove spots from an image.
```

<figure><img src="figures/spotclean_algorithm.svg" style="height:400px" align="middle"></figure>

__Parameters__

- $N$ Width of median filter.
- $k$ Threshold level for outlier detection.

### Spot cleaning - Compare performance

```{figure} figures/spotclean_compare.png
:width: 12cm

Comparing different filters to remove spots from an image.
```

The test image has some degree of noise and there are some finer edge details. When we compare the performance of the three methods to remove spots we look at both the filtered image and the difference between the original and the filtered image. Here, we can make the following observations:
- The __box filter__ with a 5x5 kernel size smooth many fine details in the image. What is also important to notice are the box shaped structures a the places of the spots. This is a consequence of the averaging characteristic of the filter type. 
- The __median filter__ still has some low pass filtering effect that cancel relevant image structures. It is however far better than the box filter in the task of rejecting outliers. 
- Finally, the __cleaning algorithm__ is the gentlest way to remove the outliers. It also leaves the remaining image untouched. 


<figure><img src="figures/spotclean_compare.svg"  style="height:500px" align="middle"></figure>

#### The ImageJ ways for outlier removal

ImageJ is a popular application for interactive image analysis. It offers two ways to remove outliers in the noise menu:

- __Despeckle__ Median ... please avoid this one!!!
- __Remove outliers__ Similar to cleaning described algorithm

## High-pass filters
High-pass filters enhance rapid changes $\rightarrow$ ideal for edge detection

### Typical high-pass filters:

<div class="row">
<div class="column">
    
#### Gradients
$$\frac{\partial}{\partial x}=\frac{1}{2}\cdot\begin{array}{|c|c|}
\hline
-1 & 1\\
\hline
\end{array}$$
    
$$\frac{\partial}{\partial x}=\frac{1}{32}\cdot\begin{array}{|c|c|c|}
\hline
-3 & 0 & 3\\
\hline
-10 & 0 & 10\\
\hline
-3 & 0 & 3\\
\hline
\end{array}$$  
        
</div>
<div class="column">
   
#### Laplacian
$$\bigtriangleup=\frac{1}{2}\cdot\begin{array}{|c|c|c|}\hline
1 & 2 & 1\\
\hline
2 & -12 & 2\\
\hline
1 & 2 & 1\\
\hline
\end{array}$$     
      
  </div>
</div>


The second gradient kernel is optimized to provide the best gradient direction estimate. Other kernels can be quite biased depending on the orientation of the edge. 

#### Sobel - Magnitude of gradient
$$
G=|\nabla f|=\sqrt{\left(\frac{\partial}{\partial x}f\right)^2 + \left(\frac{\partial}{\partial y}f\right)^2}
$$

## Gradient example - structure orientation

<div class="row">
<div class="column">

__Vertical edges__
$$\frac{\partial}{\partial x}=\frac{1}{32}\cdot\begin{array}{|c|c|c|}
\hline
-3 & 0 & 3\\
\hline
-10 & 0 & 10\\
\hline
-3 & 0 & 3\\
\hline
\end{array}$$
    
</div>
<div class="column">
 
__Horizontal edges__
$$\frac{\partial}{\partial\,y}=\frac{1}{32}\cdot\begin{array}{|c|c|c|}
\hline
-3 & -10 & -3\\
\hline
0 & 0 & 0\\
\hline
3 & 10 & 3\\
\hline
\end{array}$$
    
</div>
</div>

[Jaehne, 2005](https://doi.org/10.1007/3-540-27563-0)

In [ ]:
img=plt.imread('figures/orig.png')
k = np.array([[-3,-10,-3],[0,0,0],[3,10,3]]);
plt.figure(figsize=[12,5])
plt.subplot(1,3,1); plt.imshow(img);plt.title('Original');
plt.subplot(1,3,2); plt.imshow(ndimage.convolve(img,np.transpose(k)));plt.title('$\partial / \partial x$');
plt.subplot(1,3,3); plt.imshow(ndimage.convolve(img,k));plt.title('$\partial / \partial y$');

In this example we see that $\frac{\partial}{\partial{}x}$ enhances vertical structures while $\frac{\partial}{\partial{}y}$ enhances horizontal structures. 

We can furthermore see that the gradient image is also sensitive to whether it is a rising or falling edge seen from left to right and top to bottom. This is indicated by the dark and bright edge responses.

## Edge detection examples

Now, we have seen that the gradient filters are sensitive to edges, but that they are only able to handle one direction at the time. This is inconvenient for edge detection tasks where we want to have a single image represnting edges in any direction.

The solution to this problem is combine the two direction in a single image as in the Sobel filter

$$
G=|\nabla f|=\sqrt{\left(\frac{\partial}{\partial x}f\right)^2 + \left(\frac{\partial}{\partial y}f\right)^2}
$$

which is the magnitude of the two gradients. This provides a non-negative image with intensities proportional to the edge height. The edge would then be located at the local maximum of the edge.

A different approach is to use the a kernel based on the discretized Laplacian, i.e. the sum of the second derivatives in the principal directions.

$$\nabla^2 f = \frac{\partial{}^2 f}{\partial{}x^2} + \frac{\partial{}^2}{\partial{}y^2}$$

This edge image behaves differently; here, the midpoint of the edge is located at the sign change of the Laplacian. Thus, we have an indication of the direction the edge rises. You can see this in the figure below where the color flips from blue to red near the edges.

Note: A double scale color map was used to to visualize the sign change behaviour near the edge. When you do this it is important to set the intensity limits to +/- the same values to guarantee that the color cahnge appears at 0.

In [ ]:
img=plt.imread('figures/orig.png');
plt.figure(figsize=[15,6])
plt.subplot(1,2,2);plt.imshow(ski.filters.laplace(img),clim=[-0.08,0.08], cmap='coolwarm'); plt.title('Laplacian'); plt.colorbar(shrink=0.8);
plt.subplot(1,2,1);plt.imshow(ski.filters.sobel(img),clim=[0,0.1]); plt.title('Sobel');  plt.colorbar(shrink=0.8);

## Relevance of filters to machine learning

Convolution is a central operation in machine learning for image analysis and processing. Many of neural networks include multiple convolutions and it is part of the training process to specify the kernels.
```{figure} figures/convolutional_neural_network.png
:width: 12cm

An example of a convolutional neural network for image classification.
```

<center><img src="figures/convolutional_neural_network.png"  style="height:500px" align="middle"></center>
	
[NVIDIA Developer zone](https://developer.nvidia.com/discover/convolutional-neural-network)

# Frequency space filters

We already mentioned that the there are frequencies to filter in images. Now, we will further explore the frequency space for images. 

There is a whole collection of filters that operate directly in frequency space. 

The Fourier transform and its inverse are central operators for these filters and we will briefly show some properties of the Fourier transform the following paragraphs.  

## The Fourier transform 2D

You may only know the Fourier transform for the 1D case. It is however very easy to increase the number of dimensions. Below you see the 2D Fourier transform and its inverse. These operations are lossless one-to-one, meaning you can go from one domain to the other without loosing any information.

__Transform__

$$G(\xi_1,\xi_2)=\mathcal{F}\{g\}=\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} g(x,y)
\exp{-i(\xi_1 x+\xi_2 y)}\,dx\,dy$$

__Inverse__

$$ g(x,y)=\mathcal{F}^{-1}\{G\}=\frac{1}{(2\,\pi)^2}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} G(\xi_1,\xi_2)
\exp{i(\xi_1 x+\xi_2 y)} d\xi_1 d\xi_2$$

### FFT (Fast Fourier Transform)

The previous equations are meant for continuous signals. Images are discrete signals and the integrals turn into sums. This may introduce some numerical losses when the transforms are computed. Computing the DFT is an $O(N^2)$ operation. There is a way to speed this up by using the Fast Fourier Transform algorithm. This algorithm requires that the data has the length $N=2^k$. In this case the complexity reduces to $O(N\cdot\log(N))$, which is a radical speed-up.

In practice - you never see the transform equations.  
The Fast Fourier Transform algorithm is available in numerical libraries and tools. [Jaehne, 2005](https://doi.org/10.1007/3-540-27563-0)

## Some mathematical features of the FT
### Additition

Addition of Fourier spectra works the same way as for real space signal. The reason is that the transform is based on summarions. You probably proved this as an exercise in your math classes. The plots below shows the spectra of the sum of two signals.  

$\mathcal{F}\{a+b\} = \mathcal{F}\{a\}+\mathcal{F}\{b\}$

In [ ]:
x = np.linspace(0,50,100); s0=np.sin(0.5*x); s1=np.sin(2*x);
plt.figure(figsize=[15,5])
plt.subplot(2,3,1);plt.plot(x,s0); plt.axis('off');plt.title('$s_0$');plt.subplot(2,3,4);plt.plot(np.abs(np.fft.fftshift(np.fft.fft(s0))));plt.axis('off');
plt.subplot(2,3,2);plt.plot(x,s1); plt.axis('off');plt.title('$s_1$');plt.subplot(2,3,5);plt.plot(np.abs(np.fft.fftshift(np.fft.fft(s1))));plt.axis('off');
plt.subplot(2,3,3);plt.plot(x,s0+s1); plt.axis('off');plt.title('$s_0+s_1$');plt.subplot(2,3,6);plt.plot(np.abs(np.fft.fftshift(np.fft.fft(s0+s1))));plt.axis('off');

[1Brown3Blue about the Fourier transform](https://youtu.be/spUNpyF58BY?si=gnHf4t64zC2l8fN8)

### Convolution

Convolution, which is a relatively intense task to perform in real space reduces to a frequency-wise multiplication in the Fourier space. This is a very useful property of the transform. It allows to design some filters much easier than in real space. In particular, band stop filters to remove a specific feature in the image. Also filters with wide kernels can be faster to compute using the Fourier transform.

$\mathcal{F}\{a*b\}=\mathcal{F}\{a\}\cdot\mathcal{F}\{b\}$

alternatively

$\mathcal{F}\{a\cdot{}b\}=\mathcal{F}\{a\}*\mathcal{F}\{b\}$

## Additive noise in Fourier space

### Adding noise in real space

In [ ]:
img   = plt.imread('figures/bp_ex_original.png'); 
noise = np.random.normal(0,0.2,size=img.shape); 
nimg  = img+noise;

In [ ]:
#Visualization
fig,ax= plt.subplots(1,3,figsize=[15,3])
ax[0].imshow(img);  ax[0].set_title('Image');
ax[1].imshow(noise);ax[1].set_title('Noise');
ax[2].imshow(nimg); ax[2].set_title('Image + noise');

### Adding noise in the Fourier space

In [ ]:
fimg   = np.fft.fftshift(np.fft.fft2(img))
fnoise = np.fft.fftshift(np.fft.fft2(noise))
fnimg  = np.fft.fftshift(np.fft.fft2(nimg))

In [ ]:
#Visualization
fig,ax =plt.subplots(1,3,figsize=[15,3])

ax[0].imshow(np.abs(fimg),norm=LogNorm());
ax[0].set_title(r'| $\mathcal{F}$ Image |');
ax[1].imshow(np.abs(fnoise),norm=LogNorm());
ax[1].set_title(r'| $\mathcal{F}$ Noise |');
ax[2].imshow(np.abs(fnimg),norm=LogNorm())
ax[2].set_title(r'| $\mathcal{F}$ Image + noise |');

Looking at these spectra, you can observe the following
- The spectrum of the clean image has a cross shaped amplitude, but the intensity spreads radially in the quadrants. 
- The noise spectrum is represented by a noisy bias ampliture, which is characteristic for white noise.
- The noisy image spectrum combines the prevoius spectra. The detail we could see in the quadrant of the clean image are now obscured by the bias amplitude of the white noise. Only the larger amplitudes of the image remain visible.

A technical note: The FFT algorithm result in a cyclic representation of the Fourier transform and the quadrants of the transformed images are permuted such that $\xi_0, \xi_1=(0,0)$ is located in the top-left corner and $\xi_0, \xi_1=(-\epsilon,-\epsilon)$ is located in the lower right corner. The function ```fftshift``` permutes the quadrants to place the (0,0) in the center of the image.

### Convolution in Fourier space
How can we suppress noise without destroying relevant image features?

The filter process of image $f$:

$$f_{filtered}= \mathcal{F}^{-1}(\mathcal{F}(f)\cdot{}H)$$

#### Filter with hard threshold

$$
H(\xi_0,\xi_1)=\begin{cases}
1 & \sqrt{\xi_0^2 + \xi_1^2}<R \\
0 & \text{otherwise}
\end{cases}
$$

In this example we use a filter kernel that truncates the image spectrum above the specified frequences. It is a low-pass filter as it only keeps the lower frequencies. You can even see that the most information of the spectrum are maintained. There are only minor signal contributions outside the filter window. These are mainly related to the edges of the image which now are smoother than in the original image. 

The code below shows how the filter is implemented with the fast Fourier transform using numpy. 

In [ ]:
ff=np.fft.fftshift(np.fft.fft2(nimg))
x,y = np.meshgrid(np.linspace(-1,1,ff.shape[1]),np.linspace(-1,1,ff.shape[1]))
R=np.sqrt(x**2+y**2)
H=R<0.5
hff=np.fft.ifft2(np.fft.fftshift(H*ff)) # the inverse FFT back to the spatial domain

In [ ]:
fig,ax = plt.subplots(1,5,figsize=(15,5))
ax[0].imshow(nimg)
ax[0].set_title('Noisy image f')
ax[1].imshow(np.log(np.abs(ff)))
ax[1].set_title('log($|\mathcal{F}\{f\}|$)')
ax[2].imshow(H)
ax[2].set_title('Filter window H')
ax[3].imshow(H*np.log(np.abs(ff)))
ax[3].set_title('log($|H\cdot{}\mathcal{F}\{f\}|$)')
ax[4].imshow(np.abs(hff));
ax[4].set_title('Filtered image');

The round window filters spatial frequencies symmetrically which is good for the representations. This kind of filter is however mostly too abrubt which would be visible as ripple near the edge in less noisy images. Therefore, we explore further filter windows.

#### Filter with Gaussian window


$$
H(\xi_0,\xi_1)=\exp\left(- \frac{\xi_0^2 + \xi_1^2}{2\,\sigma^2}\right)
$$

In [ ]:
ff=np.fft.fftshift(np.fft.fft2(nimg))
x,y = np.meshgrid(np.linspace(-1,1,ff.shape[1]),np.linspace(-1,1,ff.shape[1]))
R=np.sqrt(x**2+y**2)
H=np.exp(-R**2/0.3)
hff=np.fft.ifft2(np.fft.fftshift(H*ff)) # the inverse FFT back to the spatial domain

In [ ]:
fig,ax = plt.subplots(1,5,figsize=(15,5))
ax[0].imshow(nimg)
ax[0].set_title('Noisy image f')
ax[1].imshow(np.log(np.abs(ff)))
ax[1].set_title('log($|\mathcal{F}\{f\}|$)')
ax[2].imshow(H)
ax[2].set_title('Filter window H')
ax[3].imshow(H*np.log(np.abs(ff)))
ax[3].set_title('log($|H\cdot{}\mathcal{F}\{f\}|$)')
ax[4].imshow(np.abs(hff));
ax[4].set_title('Filtered image');

The Gauss window is gentler and you can see that higher frequencies also pass through. This is the same filter we explored earlier in the real space convolution filters.

#### Filter with band pass characteristics

$$
H(\xi_0,\xi_1)=
\begin{cases}
1 & |\xi_0|<t \text{ or } |\xi_1|<t\\
0 & otherwise
\end{cases}
$$

In [ ]:
ff=np.fft.fftshift(np.fft.fft2(nimg))
x,y = np.meshgrid(np.linspace(-1,1,ff.shape[1]),np.linspace(-1,1,ff.shape[1]))
R=np.sqrt(x**2+y**2)
H=np.maximum(np.abs(x)<0.05,np.abs(y)<0.05)
hff=np.fft.ifft2(np.fft.fftshift(H*ff)) # the inverse FFT back to the spatial domain

In [ ]:
fig,ax = plt.subplots(1,5,figsize=(15,5))
ax[0].imshow(nimg)
ax[0].set_title('Noisy image f')
ax[1].imshow(np.log(np.abs(ff)))
ax[1].set_title('log($|\mathcal{F}\{f\}|$)')
ax[2].imshow(H)
ax[2].set_title('Filter window H')
ax[3].imshow(H*np.log(np.abs(ff)))
ax[3].set_title('log($|H\cdot{}\mathcal{F}\{f\}|$)')
ax[4].imshow(np.abs(hff));
ax[4].set_title('Filtered image');

### Wiener filter in Fourier space
The Wiener filter is a filter that is optimised for the noise model and pointspread function. 

$$G(u,v)=\frac{H^*(u,v)P_s(u,v)}{|H(u,v)|^2P_s(u,v)+P_n(u,v)}$$

- $H(u,v)$ Fourier transform of the point-spread function
- $P_s(u,v)$ Power spectrum of the signal process. Obtained by taking FFT of the signal autocorrelation.
- $P_n(u,v)$ Power spectrum of the noise process. Obtained by taking FFT of the noise autocorrelation.


#### Special case of the Wiener filter
In the case of additive white Gaussian noise and excluding the PSF, the Wiener filter reduces to

$$G(u,v)=\frac{P_s(u,v)}{P_s(u,v)+\sigma_n^2}$$

Here for 
- $P_s(u,v)>>\sigma^2 \Rightarrow{} G(u,v)=1$
- $P_s(u,v)<<\sigma^2 \Rightarrow{} G(u,v)=0$

__Problem__: We need to know or measure $P_s(u,v)$

#### Demonstration of the Wiener filter in AWGN

In [ ]:
fimg=np.fft.fftshift(np.fft.fft2(img))
Ps=fimg*np.conj(fimg)

var=np.prod(nimg.shape)*noise.var()
G=Ps/(Ps+1*var)
fnimg=np.fft.fftshift(np.fft.fft2(nimg))
filtered=np.real(np.fft.ifft2(np.fft.fftshift(G*fnimg)))

In [ ]:
fig,ax=plt.subplots(2,4,figsize=(15,8))
ax=ax.ravel()
ax[1].imshow(np.abs(Ps),norm=LogNorm())
ax[1].set_title('Power spectrum')
ax[2].imshow(np.abs(G),norm=LogNorm())
ax[2].set_title('Filter spectrum $G$')
ax[3].imshow(filtered)
ax[3].set_title('Filtered image')
ax[0].imshow(nimg)
ax[0].set_title('Noisy image')
ax[6].axis('off')
ax[5].axis('off')
ax[7].hist(filtered.ravel(),bins=100);
ax[4].hist(nimg.ravel(),bins=100);

In this example, you can see that the filter spectrum resembles the signal itself. This means that the relevant parts of the image will pass through the filter while regions where the noise is dominant are suppressed. The current filter introduces relativly strong smoothing, this is the consequence of the low SNR which is the quantity that controls the filter strength instead of the cut-off frequency.

## Spatial frequencies and orientation

The Fourier transform can also be used to analyse orientations in the image. This can be demonstrated using ripple images like the ones shown below. The ripple has the same spatial frequency in all examples, but the orientation of the ripple changes. 

In [ ]:
def ripple(size=128,angle=0,w0=0.1) :
    w=w0*np.linspace(0,1,size);
    [x,y]=np.meshgrid(w,w);
    img=np.sin((np.sin(angle)*x)+(np.cos(angle)*y));
    return img
N=64;
fig, ax = plt.subplots(2,4, figsize=(15,6)); ax=ax.ravel()

for idx,angle in enumerate([0,30,60,90]) :
    d=ripple(N,angle=(angle+0.1)/180*np.pi,w0=100);
    ax[idx].imshow(d); ax[idx].set_title(r'{0}'.format(angle)+r'$^{\circ}$'); ax[idx].axis('off');
    ax[idx+4].imshow((np.abs(np.fft.fftshift(np.fft.fft2(d)))));   ax[idx+4].axis('off');

The Fourier transform of the images results in two peaks to represent positive and negative spatial frequencies. All spectra have the same separation between the peaks, but they rotated to reflect the orientation of the ripple.

## Example - Stripe removal in Fourier space

The orientation sensitivity of the Fourier transform can be used to design structure matched filters. In following example we have an image with a striped background that fluctuated with time. This made it hard to flatten the image using a normalization procedure. 

So, here we try to design a matched filter to supress the slightly tilted stripes. Upon inspecting the image spectrum we can identify some plume shaped streaks near the vertical frequency axis. The ad hoc filter spectrum $H$ appears to reduce the stripes significantly without interfering too much with the image content.

- Transform the image to Fourier space
$$\mathcal{F_{2D}}\left\{a\right\} \Rightarrow A$$

- Multiply spectrum image by band pass filter
$$A_{filtered}=A \cdot H$$

- Compute the inverse transform to obtain the filtered image in real space
$$\mathcal{{F_{2D}}^{-1}}\left\{A_{filtered}\right\} \Rightarrow a_{filtered}$$

In [ ]:
plt.figure(figsize=[10,5])
plt.subplot(3,1,1);plt.imshow(plt.imread('figures/raw_img.png')); plt.title('$a$'); plt.axis('off');
plt.subplot(3,2,3);plt.imshow(plt.imread('figures/raw_spec.png')); plt.title('$\mathcal{F}(a)$'); plt.axis('off');
plt.subplot(3,2,4);plt.imshow(plt.imread('figures/filt_spec.png')); plt.title('Kernel $H$'); plt.axis('off');
plt.subplot(3,1,3);plt.imshow(plt.imread('figures/filt_img.png')); plt.title('$a_{filtered}$'); plt.axis('off');

## The effect of the stripe filter

The image in the example was a single projection from a tomography scan. Therefore, the final test is to recontruct a tomography slice to determine the improvement. The unfiltered slice has an uneven background that would interfere with the next analysis steps. 

```{figure} figures/slice_with_stripes.png
:width: 8cm

CT slice before stripe removal filter.
```

The uneven background was well to great extent reduced in the slice reconstructed from projections where the stripe filter was used. 
```{figure} figures/slice_stripe_filtered.png
:width: 8cm

CT slice after applying stripe removal filter.
```

<table><tr><td> Reconstructed CT slice before filter</td><td>Reconstructed CT slice after stripe filter</td></tr>
    <tr>
        <td><img src="figures/slice_with_stripes.png"  style="height:400px" align="middle"></td>
        <td><img src="figures/slice_stripe_filtered.png"  style="height:400px" align="middle"></td>
    <tr>
</table>

Intensity variations are suppressed using the stripe filter on all projections.

## Technical details on Fourier space filters
### When should you use convolution in Fourier space?
-  Simplicity
-  Kernel size
-  Speed at repeated convolutions

### Zero padding
The FFT is only working with data of size in $2^N$. If your data has a different length, you have to pad (fill with constant value) up the next $2^N$.

## Python functions
	
### Filters in the spatial domain
e.g. from ```scipy import ndimage```
- ```ndimage.filters.convolve(f,h)``` Linear filter using kernel $h$ on image $f$.
- ```ndimage.filters.median_filter(f,\[n,m\])```  Median filter using an $n \times m$ filter neighborhood

### Fourier transform
- ```np.fft.fft2(f)``` Computes the 2D Fast Fourier Transform of image $f$
- ```np.fft.ifft2(F)``` Computes the inverse Fast Fourier Transform $F$.
- ```np.fft.fftshift()``` Rearranges the data to center the $\omega$=0. Works for 1D and 2D.

### Complex numbers
- ```np.abs(f), np.angle(f)``` Computes amplitude and argument of a complex number.
- ```np.real(f), np.imag(f)``` Gives the real and imaginary parts of a complex number.

# Scale spaces

## Why scale spaces?

### Motivation

Image features often appear at different scales. This can be hard to access using filters on the full scaled image. The objectives of a filter optimized for tiny features may interfere with a filter desinged for large features.

Basic filters have problems to handle low SNR and textured noise.

### The solution
Filtering on different scales can take noise suppression one step further.

There are different ways to achieve the scale decomposition. A straight forward way would be to apply repeated downsampling of the image. This produces a scale pyramid like the one shown in the figure below. Here, you can see that the detailed features at level 0 are reduce to two major patches in level 4. Further downsampling does mostly not make sense as the image reduces to a constant intensity at some point. 
```{figure} figures/burt_pyramid.png
:width: 8cm

A scale pyramid of an image can be useful for filtering and segmentation.
```

<center><img src="figures/burt_pyramid.svg"  style="height:400px" align="middle"></center>

## Wavelets - the basic idea

Wavelets introduce a mathematical framework based on base functions that allow lossless transformation to and from different scales.

- The wavelet transform produces scales by decomposing a signal into two signals at a coarser scale containing __trend__ and __details__
- The next scale is computed using the trend of the previous transform

$$WT\{s\}\rightarrow\{a_1,d_1\}, WT\{a_1\}\rightarrow\{a_2,d_2\}, \ldots, WT\{a_{N-1}\}\rightarrow\{a_N,d_N\} $$

- The inverse transform brings $s$ back using $\{a_N,d_1, \ldots,d_N\}$.
- Many wavelet bases exists, the choice depends on the application.

### Applications of wavelets  
- Noise reduction 
- Analysis 
- Segmentation
- Compression
    
[Walker 2008](https://doi.org/10.1201/9781584887461) [Mallat 2009](https://doi.org/10.1016/B978-0-12-374370-1.X0001-8)

## Wavelet transform of a 1D signal

It is easier to 
```{figure} figures/wavelet1d_noaxis.png
:width: 10cm

A noisy test signal decomposed using the _symlet-4_ wavelet base function.
```

<center><img src="figures/wavelet1d_noaxis.svg" style="height:600px"></center>

Using __symlet-4__

## Wavelet transform of an image

```{figure} figures/wt2d_schematic.png
:width: 10cm

Transform workflow for a 2D wavelet decomposition.
```

<center><img src="figures/wt2d_schematic.svg" style="height:600px"></center>

## Wavelet transform of an image - example

```{figure} figures/testpattern_1024.png
:width: 8cm

Transform workflow for a 2D wavelet decomposition.
```

```{figure} figures/wt_testimage_gray.png
:width: 8cm

Transform workflow for a 2D wavelet decomposition.
```

<div class='row'>
    <div class='column'>
    
### Original
        
<img src="figures/testpattern_1024.png" style="height:500px">
    </div>
    <div class='column'>

### Wavelet transform
        
<img src="figures/wt_testimage_gray.png" style="height:500px">
    </div>
</div>
    

## Using wavelets for noise reduction
The noise is found in the detail part of the WT
-  Make a WT of the signal to a level that corresponds to the scale of the unwanted information.
- Threshold the detail part $d_{\gamma}=|d|<\gamma\quad ? \quad 0 : d$.
- Inverse WT back to normal scale $\rightarrow$ image is filtered.

```{figure} figures/wavelet1d_filter.png
:width: 10cm

The principle of a basic noise reduction filter using wavelets.
```

<center><img src="figures/wavelet1d_filter.svg" style="height:500px"></center>

## Wavelet noise reduction - Image example

Example Filtered using two levels of the Symlet-2 wavelet

```{figure} figures/wavelet2d_filtered.png
:width: 12cm

An example of noise reduction using a wavelet filter.
```

__Data:__ Neutron CT of a lead scroll
<center><img src="figures/wavelet2d_filtered.svg" style="height:600px"></center>

## Python functions for wavelets
- __dwt2__/__idtw2__ Makes one level of the wavelet transform or its inverse using wavelet base specified by 'wn'. 
- __wavedec2__ Performs N levels of wavelet decomposition using a specified wavelet base.
- __wbmpen__ Estimating threshold parameters for wavelet denoising.
- __wdencmp__ Wavelet denoising and compression using information from *wavedec2* and *wbmpen.


# Parameterized scale spaces

## PDE based scale space filters

```{figure} figures/slice_original.png
:width: 5cm

Noisy slice to be filtered.
```

```{figure} figures/slice_diffusion.png
:width: 5cm

Slice after diffusion filter.
```

```{figure} figures/slice_iss.png
:width: 5cm

Slice after ISS filter.
```


<table><tr>
    <td><img src="figures/slice_original.svg"  style="height:600px"></td>
    <td><img src="figures/slice_diffusion.svg" style="height:600px"></td>
    <td><img src="figures/slice_iss.svg"       style="height:600px"></td>
</tr></table>

These filters may work for applications where Linear and Rank filters fail.

[Kaestner et al. 2008](https://doi.org/10.1016/j.advwatres.2008.01.022)
[Aubert 2002](https://doi.org/10.1007/978-0-387-44588-5).

## The starting point
The heat transport equation
$$\frac{\partial T}{\partial t}=\kappa \nabla^2 T$$

- __$T$__ Image to filter (intensity $\equiv$ temperature)
- __$\kappa$__ Thermal conduction capacity

<table>
    <tr><td>Original</td><td>Iterations</td></tr>
    <tr>
          <td><img src="figures/lindif_iter.png" style="height:300px"></td>
          <td><video src="movies/lindif_iter.mp4" type="video/mp4" controls autoplay loop height="300px"></td>
    </tr>
</table>

<div class="alert alert-block alert-warning">
<center>The steady state solution is a homogeneous image...</center>
</div>



## Controlling the diffusivity

We want to control the diffusion process...

- _Near edges_ The Diffusivity $\rightarrow$ 0
- _Flat regions_ The Diffusivity $\rightarrow$ 1

The contrast function $G$ is our control function
$$G(x)=\frac{1}{1+\left(\frac{x}{\lambda}\right)^n}$$

- _$\lambda$_ Threshold level
- _$n$_ Steepness of the threshold function

In [ ]:
def g(x,lambd,n) :
    g=1/(1+(x/lambd)**n)
    return g

In [ ]:
x=np.linspace(0,1,100);

plt.figure(figsize=(5,3))
plt.plot(x,g(x,lambd=0.5,n=12));
plt.xlabel('Image intensity (x)'); plt.ylabel('G(x)');plt.tight_layout()

## Gradient controlled diffusivity

$$\frac{\partial u}{\partial t}=G(|\nabla u|)\nabla^2 u$$

In [ ]:
plt.subplot(1,2,1); plt.imshow(io.imread("figures/aggregates.png"),cmap='gray'); plt.title('Image');
plt.subplot(1,2,2); plt.imshow(io.imread("figures/diffusivity.png"),cmap='gray'); plt.title('Diffusivity map');

- _$u$_ Image to be filtered
- _$G(\cdot)$_ Non-linear function to control the diffusivity
- _$\tau$_ Time increment
- _$N$_ Number of iterations

<div class="alert alert-block alert-danger">
<center>This filter is noise sensitive!</center>
</div>

## The non-linear diffusion filter

A more robust filter is obtained with

$$\frac{\partial u}{\partial t}=G(|\nabla_{\sigma} u|)\nabla^2u$$

- _$u$_ Image to be filtered
- _$G(\cdot)$_ Non-linear function to control the contrast
- _$\tau$_ Time increment per numerical iteration
- _$N$_ Number of iterations
- _$\nabla_{\sigma}$_ Gradient smoothed by a Gaussian filter, width $\sigma$



## Diffusion filter example
Neutron CT slice from a real-time experiment observing the coalescence of cold mixed bitumen.

<table>    
    <tr>
        <td>Original</td><td>Iterations of non-linear diffusion</td>
    </tr>
    <tr>
        <td><img src="figures/bitumen.png" style="height:400px"></td>
        <td><video src="movies/nldif_iter.mp4" type="video/mp4" controls autoplay loop height="400px"></td>    
    </tr>
</table>


## Filtering as a regularization problem
### The continued development

- __90's__ During the late 90's the diffusion filter was described in terms of a regularization problem.
- __00's__ Work toward regularization of total variation minimization.

<div class="row">
<div class="column">

__TV-L1__
    
$$u=\underset{u\in BV(\Omega)}{\operatorname{argmin}}\left\{\underbrace{|u|_{BV}}_{noise}+ \underbrace{\mbox{$\frac{\lambda}{2}$}\|f-u\|_{1}}_{fidelity}\right\}$$
    
</div>    
<div class="column">

__TV-L2 Rudin-Osher-Fatemi model (ROF)__
    
$$u=\underset{u\in BV(\Omega)}{\operatorname{argmin}}
\left\{
\underbrace{|u|_{BV}}_{noise} 
+ \underbrace{\mbox{$\frac{\lambda}{2}$} \|f-u\|^2_{2}}_{fidelity}
\right\}$$
    
    
</div>    
</div>    

with $|u|_{BV}=\int_{\Omega}|\nabla u|^2$


## The inverse scale space filter
### The idea 
We want smooth regions with sharp edges\ldots

- Turn the processing order of scale space filter upside down
- Start with an empty image
-  Add large structures successively until an image with relevant features appears

### The ISS filter - Some properties
- is an edge preserving filter for noise reduction.
- is defined by a partial differential equation.
- has a well defined termination point.

[Burger et al. 2006](https://dx.doi.org/10.4310/CMS.2006.v4.n1.a7)

### The ROF filter equation
The image $f$ is filtered by solving

$\frac{\partial{}u}{\partial{}t}=\mathrm{div}\left(\frac{\nabla{}u}{|\nabla{}u|}\right)+\lambda{}(f-u+v)$  
$\frac{\partial{}v}{\partial{}t}=\alpha{}(f-u)$ 

<div class="row">
  <div class="column">

__Variables__      
- _$f$_ Input image
- _$u$_ Filtered image
- _$v$_ Regularization term (feedback of previous iteration)
      
  </div>
    <div class="column">
    
__Filter parameters__
- _$\lambda$_ Related to the scale of the features to suppress.
- _$\alpha$_ Quality refinement
- _$N$_ Number of iterations
- _$\tau$_ Time increment
      
  </div>    
</div>

 
 




### Filter iterations
Neutron CT of dried lung filtered using 3D ISS filter

<table>    
    <tr><td>Original</td><td>Filter iterations</td></tr>
    <tr>
    <td><img src="figures/lung_iterations_first.png" style="height:400px"></td>   
    <td><video src="movies/filter_iterations.mp4" type="video/mp4" controls autoplay loop height="400px"></td>
    </tr>
</table>


### How to choose lambda and alpha
The requirements varies between different data sets.

#### Initial conditions:
- Signal to noise ratio
- Image features (fine grained or wide spread)

#### Experiment:
- Scan $\lambda$ and $\alpha$
- Stop at $T=N\tau=\sigma$ use different $\tau$
- When does different effects occur, related to $\sigma$?

### Solutions at different times

|1|30|60|100|200|500|999|
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
|![](figures/iss-iteration_0001.png) |![](figures/iss-iteration_0030.png) |![](figures/iss-iteration_0060.png) |![](figures/iss-iteration_0100.png) |![](figures/iss-iteration_0200.png) |![](figures/iss-iteration_0500.png) |![](figures/iss-iteration_0999.png) |

<img src="figures/iss-errorplot.svg" style="height:350px">

```{figure} figures/iss-errorplot.png
:width: 8cm

Error plot for different solution times of the ISS filter.
```

### Solution time
The solution time ($T=N\tau$) is essential to the result
- _$\tau$ large_ The solution is reached fast
- _$\tau$ small_ The numerical accuracy is better

### The choice of initial image

```{figure} figures/iss-errorplots_initialimage.png
:width: 8cm

Error plots depend on the choice of the initial image.
```

<img src="figures/iss-errorplots_initialimage.svg" style="height:400px">

At some $T$ the solution with $u_0=f$ and $u_0=0$ converge.

# Non-local means

## Non-local smoothing
### The idea
Smoothing normally consider information from the neighborhood like

- Local averages (convolution)
- Gradients and Curvatures (PDE filters)

Non-local smoothing average similiar intensities in a global sense.
- Every filtered pixel is a weighted average of all pixels.
- Weights computed using difference between pixel intensities.

[Buades et al. 2005](http://dx.doi.org/10.1137/040616024)


## Filter definition
The non-local means filter is defined as
$$u(p)=\frac{1}{C(p)}\sum_{q\in\Omega}v(q)f(p,q)$$
where
- _$v$ and $u$_ input and result images.
- _$C(p)$_ is the sum of all pixel weights as    $C(p)=\sum_{q\in\Omega}f(p,q)$
    
    
- _$f(p,q)$_ is the weighting function      $f(p,q)=e^{-\frac{|B(q)-B(p)|^2}{h^2}}$
    
    
- _B(x)_ is a neighborhood operator e.g. local average around $x$

## Non-local means 2D - Example

```{figure} figures/nonlocal_example.png
:width: 10cm

Demonstration on the non-local means filter.
```

<img src="figures/nonlocal_example.png" style="height:400px">

### Observations

- Good smoothing effect.
- Strong thin lines are preserved.
-  Some patchiness related to filter parameter $t$, *i.e.* the size of $\Omega_i$.

## Performance complications
### Problem
The orignal filter compares all pixels with all pixels...
- Complexity $\mathcal{O}(N^2)$ 
- Not feasible for large images, and particular 3D images!

### Solution
It has been shown that not all pixels have to be compared to achieve a good filter effect.
i.e. $\Omega$ in the filter equations can be replaced by $\Omega_i<<\Omega$ 

## Explore the Non-local means in more detail

A notebook is prepared to study the non-local means.


# The BM3D filter

- The BM3D has evolved from the ideas in the non-local means filter. 
- Applies filters on the matching blocks instead of the average.
- It is currently one of the most efficient denoising filters.

__References__
- K. Dabov et al. 2007 [Image Denoising by Sparse 3-D Transform-Domain Collaborative Filtering](https://doi.org/10.1109/TIP.2007.901238), IEEE Trans on Image Processing, 2007
- M. LeBrun [An Analysis and Implementation of the BM3D Image Denoising Method](https://doi.org/10.5201/ipol.2012.l-bm3d), Image Processing Online, 2012
- M. Elad et al. [Image Denoising: The Deep Learning Revolution and Beyond — A Survey Paper](https://doi.org/10.1137/23M1545859), SIAM J. on Imaging Sciences, 2023

## Filter algorithm outline

![](figures/bm3d.png)


Figure from [D. Wang et al., 2020](https://doi.org/10.1109/ACCESS.2020.3006773)

### What does block matching mean?

![](figures/BM3D_BlockMatching2.png)

- Similar blocks found by a least squared difference between thresholded spectra of the images.
- The threshold is given by the image noise variance. 

Figure from [K. Dabov et al. 2007](https://doi.org/10.1109/TIP.2007.901238)

## BM3D in real images

![](figures/ComparingBM3D.png)
Figure from [Q. Zhan et al. 2024](https://www.ndt.net/search/docs.php3?id=29248)

### What about 3D images?

The BM3D filter was originally defined for 2D images.

The upscaling to 3D images is quite straight forward
- All processing steps add one dimension
- The filtering colaborative filtering step is now done on a 4D image

The filter is now called __BM4D__ instead.

#### Issues with BM4D
The additional dimension add some complexity
- Implementation: It may not be so easy to wrap your head around the 4th dimension
- Computational 
    - Memory: 3D images requires much more memory. 
    - Processing time: The increased amount of data takes longer time to process. Implementations with GPUs exist.

## Can denoising go even further?

- BM3D was long the king among denoisers
- On global scale, it is hard to get better. 
- Looking into the details there is still potential to improve

### Deep learning for denoising


```{figure} figures/DL-Denoising.png
:width: 12cm

Figure from Elad et al., https://doi.org/10.48550/arXiv.2301.03362, 2023.
```

<img src='figures/DL-Denoising.png' style="height:600px"/>
<a href="https://doi.org/10.48550/arXiv.2301.03362">Figure from Elad et al. 2023</a>

# How to choose denoiser

We have seen a collection of filters to improve the SNR. 

> You may say take the best! But what is best for your images?

There are some criteria that determines the filter type you choose.
- Do you have 2D or 3D images?
- General image composition
- Which level of detail needs to be preserved?
- How noisy are your images?
- Which filters do you already have access to?
- Do you have the hardware to use the filter?
- How much time do you have
    - To implement - Many advanced filters are not easily available.
    - To use - You can spend days on a single image, but what if you have 1000 or even more?
- etc.

Often you have to test and compare the performance of different filters to decide. 

# Enhancing and analyzing orientation
Many images have structures with dominant orientations locally and globally.

In [ ]:
balls = io.imread('figures/recon5s_0130.tif').astype(float)
hair  = io.imread('figures/hair.jpeg').astype(float).mean(axis=2)
pea   = io.imread('figures/peacock.jpeg').astype(float).mean(axis=2)

In [ ]:
#| tags: [hide-input]

fig,ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(balls,cmap='gray')
ax[0].set_title('Balls')
ax[1].imshow(hair,cmap='gray')
ax[1].set_title('Hair')
ax[2].imshow(pea,cmap='gray')
ax[2].set_title('Peakcock feather');

The question is how we can measure them...

## Gradients 

We just learned that the gradient can isolate edge information in the image. It is also computed along the principal axes of the image. Below, we see the gradient of the three test images. 

In [ ]:
#| tags: [hide-input]

fig,ax = plt.subplots(2,3,figsize=(16,9))
ax=ax.ravel()
sigma=1
ax[0].imshow(gaussian_filter(balls,sigma,order=[0,1]),cmap='gray')
ax[0].set_title(r'$\partial_x$ Balls')
ax[1].imshow(gaussian_filter(hair,sigma,order=[0,1]),cmap='gray')
ax[1].set_title(r'$\partial_x$ Hair')
t = ax[1].text(180,200,  "Vertical",
            ha="center", va="center", rotation=90, size=10,
            bbox=dict(boxstyle="rarrow,pad=0.4",
                      fc="lightblue", ec="steelblue", lw=1))
ax[2].imshow(gaussian_filter(pea,sigma,order=[0,1]),cmap='gray')
ax[2].set_title(r'$\partial_x$ Peakcock feather');

ax[3].imshow(gaussian_filter(balls,sigma,order=[1,0]),cmap='gray')
ax[3].set_title(r'$\partial_y$ Balls')
ax[4].imshow(gaussian_filter(hair,sigma,order=[1,0]),cmap='gray')
ax[4].set_title(r'$\partial_y$ Hair')
t = ax[4].text(450,200,  "Horizontal",
            ha="center", va="center", rotation=0, size=10,
            bbox=dict(boxstyle="rarrow,pad=0.4",
                      fc="lightblue", ec="steelblue", lw=1))
ax[5].imshow(gaussian_filter(pea,sigma,order=[1,0]),cmap='gray')
ax[5].set_title(r'$\partial_y$ Peakcock feather');

In these images, we clearly see that structures in different directions are enhanced. We can for example see that the vertially oriented hairs are enhanced by $\partial_x$ and the horizontally oriented hairs by $\partial_y$.

#### Going to orientations
... but we still only have image intensities. Now showing the local orientation.

In [ ]:
#| tags: [hide-input]

fig,ax = plt.subplots(1,3,figsize=(16,6))
ax=ax.ravel()
sigma=1
cmap = 'coolwarm'
aball = np.arctan2(gaussian_filter(balls,sigma,order=[0,1]),gaussian_filter(balls,sigma,order=[1,0])) 
ax[0].imshow(aball,cmap=cmap,vmin=-np.pi,vmax=np.pi)
ax[0].set_title(r'Local orientation Balls')

ahair = np.arctan2(gaussian_filter(hair,sigma,order=[0,1]),gaussian_filter(hair,sigma,order=[1,0])) 
ax[1].imshow(ahair,cmap=cmap,vmin=-np.pi,vmax=np.pi)
ax[1].set_title(r'Local orientation hair')


apea = np.arctan2(gaussian_filter(pea,sigma,order=[0,1]),gaussian_filter(pea,sigma,order=[1,0])) 
ax[2].imshow(apea,cmap=cmap,vmin=-np.pi,vmax=np.pi)
ax[2].set_title(r'Local orientation peacock feather');

## How about the Fourier transform?
We saw before that the Fourier transform can identify orientations.

In [ ]:
#| tags: [hide-input]

fig,ax=plt.subplots(1,3,figsize=(15,5))
fballs = np.fft.fftshift(np.fft.fft2(balls));
ax[0].imshow(np.abs(fballs),norm=LogNorm())
ax[0].set_title('FFT of the balls')

fhair = np.fft.fftshift(np.fft.fft2(hair));
ax[1].imshow(np.abs(fhair),norm=LogNorm())
ax[1].set_title('FFT of the hair')

fpea = np.fft.fftshift(np.fft.fft2(pea));
ax[2].imshow(np.abs(fpea),norm=LogNorm());
ax[2].set_title('FFT of the feather');

... but mainly in the global sense. Complicated images are hard to analyze.

## The structure tensor
Let's try combinations of the gradients using the outer product of the gradients

$$J(f) = \nabla{f} \cdot{} \nabla^T f= \left[ \begin{array}{cc}(\partial_x f)^2 & \partial_x f\cdot\partial_y f\\ \partial_y f\cdot\partial_x f & (\partial_y f)^2\end{array}\right]$$

Gradients are quite noisy!  
Therefore, it is common to add smoothing in the computation, e.g. using a Gaussian filter.

Using a structure tensor, also known as the second moment matrix or the inertia tensor, instead of directly using gradients to compute an orientation field, offers several advantages, particularly in the context of image analysis and computer vision. This approach is especially relevant for applications like texture analysis, edge detection, motion detection, and the analysis of local orientations in images.

### Robustness to Noise
Gradients directly computed from an image are highly sensitive to noise since differentiation amplifies high-frequency components, which include noise. The structure tensor, on the other hand, incorporates a form of averaging (usually through Gaussian smoothing) which makes the orientation field computation more robust to noise.

### Capturing Coherent Structures
The structure tensor effectively captures the dominant directions of gradient flows within a local neighborhood by aggregating the information from gradients. This aggregation helps in identifying coherent structures in regions where gradients are not consistent due to texture or noise, providing a more reliable estimate of local orientations.

### Handling Ambiguities in Orientation
Direct gradients provide a local direction of change but can be ambiguous (e.g., the gradient direction is perpendicular to an edge, not along it). The structure tensor, by considering the outer product of gradients, offers a way to resolve these ambiguities by identifying principal directions of variance in gradient orientations, which are more informative for understanding the underlying structure of the image.

### Anisotropy and Coherence Measurement
Beyond just orientation, the structure tensor can be used to measure the coherence or anisotropy of local patterns. By analyzing the eigenvalues of the structure tensor, one can distinguish between isotropic regions (where intensity changes uniformly in all directions), coherent structures (with a clear orientation), and corners (where two dominant orientations intersect).

### Scale Adaptability
The structure tensor formulation typically involves a Gaussian smoothing step, where the scale of the Gaussian kernel can be adjusted. This allows for the analysis of structures at different scales, making the method adaptable to various applications and capable of capturing features of different sizes within an image.

### Mathematical and Computational Framework
The structure tensor integrates well into the mathematical and computational frameworks used in image analysis and computer vision. It allows for a unified approach to feature extraction, and its computation can be efficiently implemented and parallelized, which is particularly beneficial for processing large images or real-time applications.

In summary, the structure tensor approach provides a more robust, informative, and versatile method for computing orientation fields in images, especially in the presence of noise, varying textures, and complex structural patterns. It enhances the reliability and accuracy of orientation estimation, which is crucial for advanced image analysis tasks.

### Implementing the structure tensor
The equation to implement the structure tensor includes a Gaussian filter, $G_{\sigma}$, with the filter with set by $\sigma$. 

$$J_{\sigma}(f) = G_{\sigma} * \nabla{f} \cdot{} \nabla^T f= \left[\begin{array}{cc}G_{\sigma} * (\partial_x f)^2 & G_{\sigma} * (\partial_x f\cdot\partial_y f)\\ G_{\sigma} * ( \partial_y f\cdot\partial_x f) & G_{\sigma} * (\partial_y f)^2\end{array}\right]$$

In [ ]:
def structure_tensor(img,sigma) :
    Ix = gaussian_filter(img,sigma,order=[0,1]) # Technical note: The Gaussian filter function can compute 
    Iy = gaussian_filter(img,sigma,order=[1,0]) # derivatives.
    J = {'J11' : Ix**2,
         'J12' : Ix*Iy,
         'J21' : Ix*Iy,
         'J22' : Iy**2} # A dict of the tensor elements
    
    return J

This function computes the structure tensor of an image and stores it in a dict having the labels J11, J12, J21, and J22. The gradients that normally would be computed using a gradient kernel is here computed together with the Gaussian smoothing of the image. This is a special feature of the Gaussian filter function provided by scipy.ndimage.

### Looking at the components of the structure tensor

In [ ]:
J = structure_tensor(balls,1.5)

In [ ]:
# Visualization
fig,axes =plt.subplots(2,2,figsize=(6,6))
axes=axes.ravel()
for ax,j in zip(axes,J) :
    m = J[j].mean()
    s = J[j].std()
    clims=[m-2*s,m+2*s]
    cmap = 'coolwarm'

    ax.imshow(J[j],clim=clims,cmap=cmap)
    ax.axis('off')
    ax.set_title(j,fontsize=9)

## Local eigen-value analysis of the structure tensor

$$\mathrm{J} \mathrm{v} = \mathrm{\lambda} \mathrm{v}$$

where $\mathrm{\lambda}$ is a diagonal matrix 
$$\mathrm{\lambda} = \left(\begin{array}{cc}\lambda_1 & 0 \\0 &\lambda_2\end{array}\right)$$


### The coherency - C

$$0\leq C = \frac{\lambda_{max}-\lambda_{min}}{\lambda_{max}+\lambda_{min}}=\frac{\sqrt{(J_{22}-J_{11})^2+4J_{12}^2}}{J_{22}+J_{11}}\leq 1$$

Measures the level of confidence of the pixel/region

$$\begin{cases}
C=0 & \lambda_{min}\approx \lambda_{max}\mbox{, the region is homogeneous, i.e. no dominant rotation.} \\
C=1 & \lambda_{min}\ll \lambda_{max}\mbox{, the rotation is well aligned with one of the axes.} \\
0<C<1 & \mbox{the local orientation lies between the axes.} 
\end{cases}$$

In [ ]:
def coherency(J,eps=0.1) :
    return np.sqrt(((J['J22']-J['J11'])**2+4*J['J12']**2))/(J['J22']+J['J11']+eps)

The epsilon in the function is a protection against division by zero in the case when $J_{11}$ and $J_{22}$ are close to zero. It should be set to a small value relative to the varaiations in the image.

### Looking at the coherency

In [ ]:
C=coherency(J,eps=(J['J22']+J['J11']).std()/10)

In [ ]:
#| tags: [hide-input]
# Visualization
fig,ax = plt.subplots(1,2,figsize=(12,4))
ax[0].imshow(balls)
ax[0].set_title('Original')
a=ax[1].imshow(C,clim=[0,1]) 
plt.colorbar(a,ax=ax[1])
ax[1].set_title('Local coherency');

Here, eps was set to 10% of the standard deviation of  $J_{11}$ and $J_{22}$.

> __Note:__ The coherency is high at the edges.

## Local orientation angle
The eigen vectors represent a rotation matrix. This allows us to compute the local orientation


$$\theta=\frac{1}{2} \arctan{\left(\frac{2 J_{12}}{J_{22}-J_{11}}\right)}$$

In [ ]:
def orientation_angle(J,eps=0.1) :
    return 0.5*np.arctan2(2*J["J12"],J["J22"]-J["J11"]+eps)

### Local orientations of the balls

In [ ]:
ang = orientation_angle(J,eps=0.01)

In [ ]:
#| tags: [hide-input]
# Visualization
fig,ax = plt.subplots(1,2,figsize=(12,4))
ax[0].imshow(balls)
ax[0].set_title('Original')
a=ax[1].imshow(ang,cmap='twilight',clim=[-np.pi/2,np.pi/2]) 
plt.colorbar(a,ax=ax[1])
ax[1].set_title('Local orientation');

## Images with line textures

The ball image has a relatively simple structure with clear edges to compute the direction of. Now, we will turn our focus to textured images with collective orientations. 

In [ ]:
#| tags: [hide-input]
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].imshow(hair,cmap='gray')
ax[0].axis('off')
ax[1].imshow(pea,cmap='gray')
ax[1].axis('off');

### Struture tensor the hair image

In [ ]:
Jh = structure_tensor(hair,sigma=0.75)
Ch = coherency(Jh,eps=0.05)
Ah = orientation_angle(Jh,eps=0.1)

In [ ]:
fig,axes = plt.subplots(2,3,figsize=(15,6))
axes = axes.ravel()

axes[0].imshow(hair,cmap='gray')
axes[1].imshow(Ch)
a2=axes[2].imshow(Ah,cmap='twilight',clim=[-np.pi/2,np.pi/2])
fig.colorbar(a2,ax=axes[2])
axes[3].hist(hair.ravel(),bins=200);
axes[4].hist(Ch.ravel(),bins=200);
axes[5].hist(Ah.ravel(),bins=200);

### Structure tensor of the peacock feather

In [ ]:
Jp = structure_tensor(pea,sigma=0.75)
Cp = coherency(Jp,eps=1)
Ap = orientation_angle(Jp,eps=0.1)

In [ ]:
#| tags: [hide-input]
fig,axes = plt.subplots(2,3,figsize=(13,6))
axes = axes.ravel()

axes[0].imshow(pea,cmap='gray')
axes[1].imshow(Cp)
a2=axes[2].imshow(Ap,cmap='twilight',clim=[-np.pi/2,np.pi/2])
fig.colorbar(a2,ax=axes[2])
axes[3].hist(pea.ravel(),bins=200);
axes[4].hist(Cp.ravel(),bins=200);
axes[5].hist(Ap.ravel(),bins=200);

## Compare the orientations globally

In [ ]:
#| tags: [hide-input]
fig,ax = plt.subplots(2,2,figsize=(12,8))
ax=ax.ravel()
ax[0].imshow(hair,cmap='gray')
t = ax[0].text(300,200,  "Direction",
            ha="center", va="center", rotation=0, size=15,
            bbox=dict(boxstyle="rarrow,pad=0.3",
                      fc="lightblue", ec="steelblue", lw=2))
ax[1].imshow(pea,cmap='gray');
t = ax[1].text(400,370,  "Direction",
            ha="center", va="center", rotation=40, size=15,
            bbox=dict(boxstyle="rarrow,pad=0.3",
                      fc="lightblue", ec="steelblue", lw=2))

hh=ax[2].hist(Ah.ravel(),bins=200);
hhm=medfilt(hh[0],9)
ap=hh[1][np.argmax(hhm)]
ax[2].axvline(x=ap, color='r')
ax[2].set_xticks([-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2])
ax[2].set_xticklabels([r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{4}$',0,r'$\frac{\pi}{4}$',r'$\frac{\pi}{2}$']);

hp=ax[3].hist(Ap.ravel(),bins=200);
hpm=medfilt(hp[0],9)
ap=hp[1][np.argmax(hpm)]
ax[3].axvline(x=ap, color='r')
ax[3].set_xticks([-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2])
ax[3].set_xticklabels([r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{4}$',0,r'$\frac{\pi}{4}$',r'$\frac{\pi}{2}$']);


## Compare orientations locally

In both cases, we see that the global orientation is unable to capture that there are local variations. Now, let's take a look at the local histograms in some regions of the peacock feather.

In [ ]:
#| tags: [hide-input]
fig,ax = plt.subplots(2,3,figsize=(15,8))
ax=ax.ravel()
ax[0].axis('off')
ax[2].axis('off')
ax[1].imshow(Ap,cmap='twilight')

ax[3].hist(Ap[400:500,0:100].ravel(),bins=100);
ax[3].set_xticks([-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2])
ax[3].set_xticklabels([r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{4}$',0,r'$\frac{\pi}{4}$',r'$\frac{\pi}{2}$'])

ax[4].hist(Ap[550:650,200:300].ravel(),bins=100);
ax[4].set_xticks([-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2])
ax[4].set_xticklabels([r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{4}$',0,r'$\frac{\pi}{4}$',r'$\frac{\pi}{2}$'])

ax[5].hist(Ap[150:250,550:650].ravel(),bins=100);
ax[5].set_xticks([-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2])
ax[5].set_xticklabels([r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{4}$',0,r'$\frac{\pi}{4}$',r'$\frac{\pi}{2}$'])



rect3 = plt.Rectangle((0,400), 100, 100, facecolor="yellow", alpha=0.5,ec="yellow")
ax[1].add_patch(rect3)



rect4 = plt.Rectangle((200,550), 100, 100,
                     facecolor="yellow", alpha=0.5,ec="yellow")
ax[1].add_patch(rect4)

rect5 = plt.Rectangle((550,150), 100, 100, facecolor="yellow", alpha=0.5,ec="yellow")
ax[1].add_patch(rect5)



con3 = ConnectionPatch(xyA=(0,150), xyB=(50,450), 
                       coordsA="data", coordsB="data", 
                       axesA=ax[3], axesB=ax[1], 
                       color="crimson", lw=3)
ax[1].add_artist(con3)

con4 = ConnectionPatch(xyA=(0,150), xyB=(250,600), 
                       coordsA="data", coordsB="data", 
                       axesA=ax[4], axesB=ax[1], 
                       color="crimson", lw=3)
ax[1].add_artist(con4)


con5 = ConnectionPatch(xyA=(0,250), xyB=(600,200), 
                       coordsA="data", coordsB="data", 
                       axesA=ax[5], axesB=ax[1], 
                       color="crimson", lw=3)
ax[1].add_artist(con5);

### Quantifying local orientations
It is mostly not relevant to observe the orientation per pixel, but rather in a region.

How can we get the region information
- We saw that there is a uniformly distributed bias in the angles
- The avagerage will be misleading, use max of histogram.

In [ ]:
def local_angle(img, w) :
    res=np.zeros([1+img.shape[0]//w,1+img.shape[1]//w])

    for rr,r in enumerate(np.arange(0,img.shape[0],w)) :
        for rc,c in enumerate(np.arange(0,img.shape[1],w)) :
            h,a = np.histogram(img[r:r+w,c:c+w].ravel(),bins=w)
            h=medfilt(h)
            res[rr,rc]= a[np.argmax(h)]
    return res

### Compute the local orientation on the peacock feather

In [ ]:
# Compute the average angle in a window w
w=100
Ap_avg = local_angle(Ap,w)

# Compute positions
r,c = np.meshgrid(np.arange(0,Ap_avg.shape[0])*w+w/2,np.arange(0,Ap_avg.shape[1]-1)*w+w/2)
# Compute the vectors
u = np.cos(Ap_avg[:,:-1])
v = np.sin(Ap_avg[:,:-1])

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,3.5))

a = ax[0].imshow(Ap_avg,clim=[-np.pi/2, np.pi/2],cmap='twilight')
ax[0].set_title('Local orientation angles')
fig.colorbar(a,ax=ax[0])
m,s=pea.mean(), pea.std()
a1=ax[1].imshow(pea,clim=[m-2*s,m+2*s])
fig.colorbar(a1,ax=ax[1])
ax[1].quiver( c,r, u,v, color='red');
ax[1].set_title('Orientations as vector field');


# Verification
## How good is my filter?

## Verify the correctness of the method
### "Data massage"
Filtering manipulates the data, avoid too strong modifications otherwise you may invent new image features!!!

```{figure} figures/mugs.png
:width: 8cm


"Don't trust that guy, he makes cup of us all." -- Be careful when you apply different filters. You may make to great modifications.
```

<figure><img src="figures/mugs.png" style="height:250px">
<figcaption>"Don't trust that guy, he makes cup of us all."</figcaption>
</figure>

### Verify the validity your method
- Visual inspection
- Difference images
- Use degraded phantom images in a "smoke test"

## Verification using difference images
Compute pixel-wise difference between image $f$ and $g$
<img src="figures/verification_differences.svg" style="height:600px">

Difference images provide first diagnosis about processing performance 

```{figure} figures/verification_differences.png
:width: 12cm


The performance of filters can be verified using test patterns.
```

## Performance testing - "The smoke test"

- Testing term from electronic hardware testing - drive the system until something fails due to overheating...
- In general: scan the parameter space for different SNR until the method fails to identify strength and weakness of the system.

### Test strategy
1. Create a phantom image with relevant features.
2. Add noise for different SNR to the phantom.
3. Apply the processing method with different parameters.
4. Measure the difference between processed and phantom.
5. Repeat steps 2-4 $N$ times for better test statistics.
6. Plot the results and identify the range of SNR and parameters that produce acceptable results.

## Data for evaluation -  Phantom data
General purpose can be controlled
- Data with known features.
- Parameters can be changed.
    - Shape
    - Sharpness
    - Contrast
    - Noise (distribution and strength)
<table><tr>
<td><img src="figures/shepplogan.png" style="height:250px"</td>
<td><img src="figures/synthetic_root4.png" style="height:250px"</td>
    </tr></table>

```{figure} figures/shepplogan.png
:width: 8cm
The  shepp logan phantom is often used for computed tomography reconstruction tests.
```

```{figure} figures/synthetic_root4.png
:width: 8cm

A simulated root network for testing a full workflow including filtering and segmentation.
```

## Data for evaluation - Labelled data

<div class="row">
  <div class="column23">
      
Often 'real' data
- Labeled by experts
- Used for training and validation
     - Training of model
     - Validation 
     - Test
      
   </div>
   <div class="column13"> 
    <img src="figures/mnist.png" style="height:400px">
   </div>
</div>



```{figure} figures/mnist.png
:width: 8cm

Images of hand written numbers from the MNIST data base.
```

## Evaluation metrics for gray level images
An evaluation procedure needs a metric to compare the performance
### Mean squared error
<font size=5>
    
$$MSE(f,g)=\sum_{p\in \Omega}(f(p)-g(p))^2$$
    
</font>

### Structural similarity index
<font size=5>
    
$$SSIM(f,g)=\frac{(2\mu_f\,\mu_g+C_1)(2\sigma_{fg}+C_2)}{(\mu_f^2+\mu_g^2+C_1)(\sigma_f^2+\sigma_g^2+C_2)}$$

</font>
    
- _$\mu_f$, $\mu_g$_ Local mean of $f$ and $g$.
- _$\sigma_{fg}$_ Local correlation between $f$ and $g$.
- _$\sigma_f$, $\sigma_g$_ Local standard deviation of $f$ and $g$.
- _$C_1$, $C_2$_ Constants based on the image dynamics (small numbers).


<font size=5>
    
$$MSSIM(f,g)=E[SSIM(f,g)]$$

</font>

[Wang 2009](https://doi.org/10.1109/MSP.2008.930649)

## Test run example

Runnig tests with different structure sizes and SNR. 

### Phantom structure sizes
|1|2|4|8|
|:---:|:---:|:---:|:---:|
|![](figures/synthetic_root1.png)|![](figures/synthetic_root2.png)|![](figures/synthetic_root4.png)|![](figures/synthetic_root8.png)|

### Change SNR and contrast

| $\sigma$=1|$\sigma$=2|$\sigma$=5|$\sigma$=10|
|:---:|:---:|:---:|:---:|
|![](figures/noise01.png)|![](figures/noise02.png)|![](figures/noise05.png)|![](figures/noise10.png)|

### Process

<div class="alert alert-block alert-success">
Apply processing sequence.
</div>

### Plot results
| R=1|R=2|R=4|R=8|
|:---:|:---:|:---:|:---:|
|![](figures/radius1.png)|![](figures/radius2.png)|![](figures/radius4.png)|![](figures/radius8.png)|

[Kaestner et al. 2006](https://doi.org/10.1016/j.geoderma.2006.04.009)

# Overview

## Many filters

```{figure} figures/filter_overview_gray.png
:width: 12cm

All filters form the lecture for different SNR.
```

<img src="figures/filter_overview_gray.svg" style="height:600px">

## Details of filter performance

```{figure} figures/filter_overview_close_gray.png
:width: 12cm

A close-up of all filters form the lecture for different SNR.
```

<img src="figures/filter_overview_close_gray.svg" style="height:600px">

## Recommended Strategy in Practice
- Start with simple Gaussian filter
- Evaluate bias on phantom
- Check boundary shift
- Quantify variance reduction
- Only then consider advanced filters

Advanced methods should justify their complexity.

## Image enhancement is not cosmetic.

Filtering changes:
- Noise variance
- Spatial resolution
- Bias of measurements
- Detectability of features


In quantitative imaging we must ask:

> Does filtering improve the measurement — or distort it?

## Take-home message
We have looked at different ways to suppress noise and artifacts:

- Convolution 
- Median filters
- Wavelet denoising
- PDE filters

Don't ask "Which filter is best?"

Instead ask:
- What is the measurement target?
- What spatial scale matters?
- What bias is acceptable?
- Is computation time limiting?

<div class="alert alert-block alert-success">
<b>Remember</b>: A good measurement is better than an enhanced bad measurement, but bad data can mostly be rescued if needed.
</div>

Filters can also be used to provide quantitative information

